# RAG-Anything Tutorial

This notebook demonstrates how to use RAG-Anything for multimodal document processing and retrieval.

## Prerequisites
- Docker container running with RAG-Anything
- OpenAI API key set in environment
- Sample documents in `/app/inputs/`

## 1. Setup and Initialization

In [ ]:
# Import required libraries
import os
import sys
import asyncio
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

# Add parent directory to path
sys.path.append('/app')

# Import RAG-Anything
from raganything import RAGAnything, RAGAnythingConfig
from lightrag.llm.openai import openai_complete_if_cache, openai_embed
from lightrag.utils import EmbeddingFunc

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

print("✅ Libraries imported successfully!")

In [ ]:
# Check environment
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("✅ OpenAI API key found")
else:
    print("❌ OpenAI API key not found. Please set OPENAI_API_KEY environment variable")

# Check available documents
input_dir = Path("/app/inputs")
if input_dir.exists():
    files = list(input_dir.glob("*"))
    print(f"\n📁 Found {len(files)} files in input directory:")
    for f in files[:5]:  # Show first 5 files
        print(f"  - {f.name}")
    if len(files) > 5:
        print(f"  ... and {len(files) - 5} more")

## 2. Initialize RAG-Anything

In [ ]:
# Configuration
config = RAGAnythingConfig(
    working_dir="/app/rag_storage",
    parser="mineru",
    parse_method="auto",
    enable_image_processing=True,
    enable_table_processing=True,
    enable_equation_processing=True,
)

# Display configuration
print("Configuration:")
print(f"  Parser: {config.parser}")
print(f"  Image processing: {config.enable_image_processing}")
print(f"  Table processing: {config.enable_table_processing}")
print(f"  Equation processing: {config.enable_equation_processing}")

In [ ]:
# Define model functions
base_url = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")

def llm_model_func(prompt, system_prompt=None, history_messages=[], **kwargs):
    return openai_complete_if_cache(
        "gpt-4o-mini",
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=api_key,
        base_url=base_url,
        **kwargs,
    )

def vision_model_func(prompt, system_prompt=None, history_messages=[], image_data=None, **kwargs):
    if image_data:
        return openai_complete_if_cache(
            "gpt-4o",
            "",
            messages=[
                {"role": "system", "content": system_prompt} if system_prompt else None,
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}}
                    ],
                }
            ],
            api_key=api_key,
            base_url=base_url,
            **kwargs,
        )
    return llm_model_func(prompt, system_prompt, history_messages, **kwargs)

embedding_func = EmbeddingFunc(
    embedding_dim=3072,
    max_token_size=8192,
    func=lambda texts: openai_embed(
        texts,
        model="text-embedding-3-large",
        api_key=api_key,
        base_url=base_url,
    ),
)

print("✅ Model functions defined")

In [ ]:
# Initialize RAG-Anything
rag = RAGAnything(
    config=config,
    llm_model_func=llm_model_func,
    vision_model_func=vision_model_func,
    embedding_func=embedding_func,
)

print("✅ RAG-Anything initialized successfully!")

## 3. Process Documents

In [ ]:
# Process a single document
# Change this to your actual document path
document_path = "/app/inputs/sample.pdf"

if Path(document_path).exists():
    print(f"Processing: {document_path}")
    
    # Process document asynchronously
    await rag.process_document_complete(
        file_path=document_path,
        output_dir="/app/output",
        parse_method="auto"
    )
    
    print("✅ Document processed successfully!")
else:
    print(f"❌ Document not found: {document_path}")
    print("Please update the document_path variable with an existing file.")

In [ ]:
# Process multiple documents (batch processing)
batch_folder = "/app/inputs"

if Path(batch_folder).exists():
    print(f"Processing all documents in: {batch_folder}")
    
    await rag.process_folder_complete(
        folder_path=batch_folder,
        output_dir="/app/output",
        file_extensions=[".pdf", ".docx", ".pptx"],
        recursive=False,
        max_workers=2
    )
    
    print("✅ Batch processing completed!")

## 4. Query Documents

In [ ]:
# Simple text query
query = "What are the main topics discussed in the document?"

print(f"Query: {query}")
result = await rag.aquery(query, mode="hybrid")

display(Markdown(f"### Answer:\n{result}"))

In [ ]:
# Multiple queries
queries = [
    "What are the key findings?",
    "What methodology was used?",
    "What are the conclusions?",
    "What are the recommendations?"
]

results = {}
for q in queries:
    print(f"\n📝 Querying: {q}")
    answer = await rag.aquery(q, mode="hybrid")
    results[q] = answer
    display(Markdown(f"**Answer:** {answer[:200]}..."))

## 5. Multimodal Queries

In [ ]:
# Query with table data
table_data = """
| Metric | Before | After | Improvement |
|--------|--------|-------|-------------|
| Speed  | 100ms  | 50ms  | 50%         |
| Memory | 500MB  | 300MB | 40%         |
| CPU    | 80%    | 40%   | 50%         |
"""

multimodal_query = "How do these performance improvements compare to what's discussed in the document?"

result = await rag.aquery_with_multimodal(
    multimodal_query,
    multimodal_content=[{
        "type": "table",
        "table_data": table_data,
        "table_caption": "System Performance Improvements"
    }],
    mode="hybrid"
)

display(Markdown(f"### Multimodal Query Result:\n{result}"))

In [ ]:
# Query with equation
equation_query = "Explain this formula in the context of the document"

result = await rag.aquery_with_multimodal(
    equation_query,
    multimodal_content=[{
        "type": "equation",
        "latex": r"F = G \frac{m_1 m_2}{r^2}",
        "equation_caption": "Newton's Law of Universal Gravitation"
    }],
    mode="hybrid"
)

display(Markdown(f"### Equation Analysis:\n{result}"))

## 6. Direct Content Insertion

In [ ]:
# Create custom content list
custom_content = [
    {
        "type": "text",
        "text": "This is a custom document created directly in the notebook.",
        "page_idx": 0
    },
    {
        "type": "text",
        "text": "It demonstrates how to insert content without parsing files.",
        "page_idx": 0
    },
    {
        "type": "table",
        "table_body": """| Feature | Status |
|---------|--------|
| Direct insertion | ✅ Supported |
| Multimodal | ✅ Supported |
| Fast processing | ✅ Supported |""",
        "table_caption": ["Feature Support Matrix"],
        "page_idx": 1
    },
    {
        "type": "equation",
        "latex": "E = mc^2",
        "text": "Einstein's famous equation",
        "page_idx": 1
    }
]

# Insert the content
await rag.insert_content_list(
    content_list=custom_content,
    file_path="notebook_generated.pdf",
    display_stats=True
)

print("✅ Custom content inserted successfully!")

In [ ]:
# Query the custom content
custom_query = "What features are supported according to the notebook content?"
result = await rag.aquery(custom_query, mode="hybrid")

display(Markdown(f"### Custom Content Query Result:\n{result}"))

## 7. Visualization and Analysis

In [ ]:
# Create a simple visualization of query results
if results:
    # Word count analysis
    word_counts = {q: len(a.split()) for q, a in results.items()}
    
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(word_counts)), list(word_counts.values()))
    plt.xticks(range(len(word_counts)), [q[:20] + "..." for q in word_counts.keys()], rotation=45)
    plt.ylabel("Word Count")
    plt.title("Response Length by Query")
    plt.tight_layout()
    plt.show()
    
    # Create results dataframe
    df = pd.DataFrame([
        {"Query": q, "Response Length": len(a), "Word Count": len(a.split())}
        for q, a in results.items()
    ])
    display(df)

## 8. Advanced Features

In [ ]:
# Query with different modes
modes = ["hybrid", "local", "global", "naive"]
test_query = "What is the main purpose?"

mode_results = {}
for mode in modes:
    try:
        result = await rag.aquery(test_query, mode=mode)
        mode_results[mode] = result
        print(f"\n📊 Mode: {mode}")
        print(f"Result: {result[:100]}...")
    except Exception as e:
        print(f"Error with mode {mode}: {e}")

In [ ]:
# Custom modal processor example
from raganything.modalprocessors import GenericModalProcessor

class NotebookModalProcessor(GenericModalProcessor):
    async def process_multimodal_content(self, modal_content, content_type, file_path, entity_name):
        if content_type == "notebook_cell":
            code = modal_content.get("code", "")
            output = modal_content.get("output", "")
            
            description = f"Jupyter notebook cell containing: {code[:50]}..."
            if output:
                description += f" with output: {output[:50]}..."
            
            entity_info = {
                "entity_name": entity_name,
                "entity_type": "notebook_cell",
                "code": code,
                "output": output
            }
            
            return await self._create_entity_and_chunk(description, entity_info, file_path)

print("✅ Custom processor defined")

## 9. Best Practices and Tips

In [ ]:
# Error handling example
async def safe_query(rag_instance, query, retries=3):
    """Safely execute a query with retries"""
    for attempt in range(retries):
        try:
            result = await rag_instance.aquery(query)
            return result
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt == retries - 1:
                raise
            await asyncio.sleep(1)  # Wait before retry

# Test safe query
try:
    result = await safe_query(rag, "What are the main points?")
    print("Query successful!")
except Exception as e:
    print(f"Query failed after retries: {e}")

In [ ]:
# Performance monitoring
import time

async def benchmark_query(rag_instance, queries, mode="hybrid"):
    """Benchmark query performance"""
    times = []
    
    for query in queries:
        start = time.time()
        await rag_instance.aquery(query, mode=mode)
        elapsed = time.time() - start
        times.append(elapsed)
    
    avg_time = sum(times) / len(times)
    print(f"Average query time: {avg_time:.2f} seconds")
    print(f"Min time: {min(times):.2f}s, Max time: {max(times):.2f}s")
    
    return times

# Run benchmark
test_queries = [
    "What is this about?",
    "Summarize the content",
    "What are the key points?"
]

times = await benchmark_query(rag, test_queries)

# Plot results
plt.figure(figsize=(8, 4))
plt.plot(times, marker='o')
plt.xlabel("Query Number")
plt.ylabel("Time (seconds)")
plt.title("Query Performance")
plt.grid(True)
plt.show()

## 10. Cleanup and Summary

In [ ]:
# Summary of what we've done
summary = f"""
## RAG-Anything Tutorial Summary

✅ **Initialized** RAG-Anything with multimodal support
✅ **Processed** documents using MinerU parser
✅ **Executed** text queries in different modes
✅ **Performed** multimodal queries with tables and equations
✅ **Inserted** custom content directly
✅ **Demonstrated** error handling and performance monitoring

### Key Capabilities:
- Document parsing (PDF, DOCX, PPTX, etc.)
- Image analysis with vision models
- Table processing and understanding
- Mathematical equation handling
- Flexible query modes (hybrid, local, global, naive)
- Batch processing for multiple documents
- Direct content insertion without files

### Next Steps:
1. Try with your own documents
2. Experiment with different query modes
3. Create custom modal processors
4. Build applications using the API client
"""

display(Markdown(summary))

In [ ]:
# Save results to file
output_file = "/app/output/tutorial_results.json"

tutorial_results = {
    "queries": results if 'results' in locals() else {},
    "mode_comparison": mode_results if 'mode_results' in locals() else {},
    "performance": {
        "times": times if 'times' in locals() else [],
        "average": sum(times)/len(times) if 'times' in locals() and times else 0
    }
}

with open(output_file, 'w') as f:
    json.dump(tutorial_results, f, indent=2)

print(f"✅ Results saved to: {output_file}")